# 四模型全面对比: SenseVoice & Fun-ASR-Nano (基线 vs 微调)

在相同测试集上并行评估四个模型：
1. **SenseVoice-base**: 未微调基线
2. **Nano-base**: 未微调基线
3. **Nano-finetuned**: 微调后
4. **SenseVoice-finetuned**: 微调后

## 1. 环境检查

In [ ]:
import os, re, json, time, gc
import torch
from concurrent.futures import ThreadPoolExecutor

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

try:
    import funasr
    print(f"FunASR: {funasr.__version__}")
except ImportError:
    print("FunASR 未安装")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("matplotlib 未安装")

print("\n--- 路径检查 ---")
paths = {
    "SenseVoice 微调模型": "/mnt/data/output/sensevoice_lora/model.pt.best",
    "Nano 微调 checkpoint": "/mnt/output/funasr_nano_v3/model.pt.best",
    "测试集": "/mnt/data/prepared_data/sensevoice/val.jsonl",
    "音频目录": "/mnt/data/hengdong_asr_trainset/audio",
}
for name, path in paths.items():
    print(f"  {name}: {'✓' if os.path.exists(path) else '✗'} {path}")

## 2. 工具函数与数据加载

In [ ]:
import threading

# Thread-safe progress tracking
_progress = {}
_progress_lock = threading.Lock()
_monitor_done = threading.Event()

def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def clean_sensevoice_text(text):
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU 显存已释放, 当前: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

def eval_sensevoice(model, samples, label):
    """评估 SenseVoice 模型（线程安全，无打印输出）"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    for i, s in enumerate(samples):
        audio, expected = s['source'], s['target']
        if not os.path.exists(audio):
            continue
        try:
            res = model.generate(input=audio, language="auto", use_itn=True)
            pred = clean_sensevoice_text(res[0]['text']) if res else ""
        except:
            pred = ""
        dist = levenshtein(expected, pred)
        ref_len = max(len(expected), 1)
        cer = dist / ref_len
        total_cer += dist
        total_chars += ref_len
        if expected == pred:
            exact += 1
        results.append({"id": i, "expected": expected, "predicted": pred, "cer": cer})
        with _progress_lock:
            _progress[label] = {
                "done": len(results), "total": len(samples),
                "cer": total_cer / total_chars if total_chars > 0 else 0,
                "exact": exact,
            }
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    return {"name": label, "results": results, "cer": cer, "exact": exact, "total": len(results), "time": elapsed}

def eval_nano(model, samples, label):
    """评估 Fun-ASR-Nano 模型（线程安全，无打印输出）"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio, expected = s['source'], s['target']
            if not os.path.exists(audio):
                continue
            try:
                res = model.generate(input=audio, language="中文", itn=True, max_length=64)
                pred = res[0]['text'] if res else ""
            except:
                pred = ""
            dist = levenshtein(expected, pred)
            ref_len = max(len(expected), 1)
            cer = dist / ref_len
            total_cer += dist
            total_chars += ref_len
            if expected == pred:
                exact += 1
            results.append({"id": i, "expected": expected, "predicted": pred, "cer": cer})
            with _progress_lock:
                _progress[label] = {
                    "done": len(results), "total": len(samples),
                    "cer": total_cer / total_chars if total_chars > 0 else 0,
                    "exact": exact,
                }
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    return {"name": label, "results": results, "cer": cer, "exact": exact, "total": len(results), "time": elapsed}

# 加载测试集（SenseVoice 格式: source + target）
TEST_PATH = "/mnt/data/prepared_data/sensevoice/val.jsonl"
samples = []
with open(TEST_PATH) as f:
    for line in f:
        samples.append(json.loads(line))

valid_samples = [s for s in samples if os.path.exists(s['source'])]
print(f"测试集: {len(samples)} 条, 有效: {len(valid_samples)} 条")

## 3. 四模型并行评估

同时加载 4 个模型，ThreadPoolExecutor 4 线程并行推理，最大化 GPU 利用率。

In [ ]:
from funasr import AutoModel
import sys

# Fun-ASR-Nano 需要仓库路径来注册模型类
FUNASR_REPO = "/mnt/Fun-ASR"
if os.path.exists(FUNASR_REPO) and FUNASR_REPO not in sys.path:
    sys.path.insert(0, FUNASR_REPO)
    print(f"已添加 Fun-ASR 仓库路径: {FUNASR_REPO}")

REPORT_DIR = "/mnt/output"
SV_DIR = "/mnt/data/output/sensevoice_lora"
SV_CKPT = f"{SV_DIR}/model.pt.best"
NANO_DIR = "/mnt/output/funasr_nano_v3"

# --- 加载所有 4 个模型 ---
print("加载模型...")
print("  [1/4] SenseVoice-base")
sv_base = AutoModel(model="iic/SenseVoiceSmall", disable_update=True)
print("  [2/4] Nano-base")
nano_base = AutoModel(model="FunAudioLLM/Fun-ASR-Nano-2512", trust_remote_code=True, hub="ms")
nano_base.model.eval()
print("  [3/4] Nano-ft")
nano_ft = AutoModel(
    model="FunAudioLLM/Fun-ASR-Nano-2512",
    trust_remote_code=True, hub="ms",
    init_param=f"{NANO_DIR}/model.pt.best",
)
nano_ft.model.eval()
print("  [4/4] SenseVoice-ft")
sv_ft = AutoModel(model="iic/SenseVoiceSmall", lora_only=True, disable_update=True)
ckpt = torch.load(SV_CKPT, map_location="cpu")
print(f"  Key 匹配: {len(set(sv_ft.model.state_dict().keys()) & set(ckpt['state_dict'].keys()))}/{len(sv_ft.model.state_dict())}")
sv_ft.model.load_state_dict(ckpt["state_dict"], strict=False)
del ckpt
print("全部模型加载完成!\n")

# --- 线程级输出过滤：吞掉评估线程的 tqdm，只放行监控线程 ---
class _FilteredOutput:
    def __init__(self, original):
        self._original = original
        self._allowed = set()
        self._lock = threading.Lock()
    def allow(self):
        with self._lock:
            self._allowed.add(threading.current_thread().ident)
    def write(self, data):
        if threading.current_thread().ident in self._allowed:
            return self._original.write(data)
        return len(data)
    def flush(self):
        self._original.flush()
    def __getattr__(self, name):
        return getattr(self._original, name)

_real_stdout, _real_stderr = sys.stdout, sys.stderr
sys.stdout = _FilteredOutput(_real_stdout)
sys.stderr = _FilteredOutput(_real_stderr)

# --- 进度监控线程 ---
MODEL_LABELS = ["SV-base", "Nano-base", "Nano-ft", "SV-ft"]

def _progress_monitor():
    sys.stdout.allow()
    sys.stderr.allow()
    while not _monitor_done.is_set():
        _monitor_done.wait(3)
        with _progress_lock:
            parts = []
            for label in MODEL_LABELS:
                p = _progress.get(label, {})
                if p:
                    parts.append(f"{label}: {p['done']}/{p['total']} CER={p['cer']:.1%}")
        if parts:
            print(" | ".join(parts))

_monitor_done.clear()
monitor_thread = threading.Thread(target=_progress_monitor, daemon=True)
monitor_thread.start()

# --- 并行评估 4 个模型 ---
total_start = time.time()

def _safe_eval(fn, model, samples, label):
    try:
        return fn(model, samples, label)
    except Exception as e:
        import traceback
        _real_stdout.write(f"[ERROR] {label}: {e}\n{traceback.format_exc()}\n")
        _real_stdout.flush()
        return None

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {
        pool.submit(_safe_eval, eval_sensevoice, sv_base, valid_samples, "SV-base"): "SV-base",
        pool.submit(_safe_eval, eval_nano, nano_base, valid_samples, "Nano-base"): "Nano-base",
        pool.submit(_safe_eval, eval_nano, nano_ft, valid_samples, "Nano-ft"): "Nano-ft",
        pool.submit(_safe_eval, eval_sensevoice, sv_ft, valid_samples, "SV-ft"): "SV-ft",
    }
    result_map = {}
    for future in futures:
        label = futures[future]
        result_map[label] = future.result()

# 停止监控并恢复输出
_monitor_done.set()
monitor_thread.join(timeout=5)
sys.stdout, sys.stderr = _real_stdout, _real_stderr

# --- 整理结果 ---
sv_base_res = result_map.get("SV-base")
nano_base_res = result_map.get("Nano-base")
nano_ft_res = result_map.get("Nano-ft")
sv_ft_res = result_map.get("SV-ft")
all_results = [r for r in [sv_base_res, nano_base_res, nano_ft_res, sv_ft_res] if r]

total_elapsed = time.time() - total_start

# 释放 GPU
del sv_base, nano_base, nano_ft, sv_ft
free_gpu()

print(f"\n{'='*60}")
print(f"四模型评估完成! 总耗时: {total_elapsed:.1f}s")
print(f"{'='*60}")
for r in all_results:
    print(f"  {r['name']}: CER={r['cer']:.2%}, 精确={r['exact']}/{r['total']}, 耗时={r['time']:.1f}s")

## 4. 汇总对比

In [ ]:
all_results = [sv_base_res, nano_base_res, nano_ft_res, sv_ft_res]

print("=" * 70)
print("四模型对比汇总")
print("=" * 70)
print(f"{'指标':<16} {'SV-base':>10} {'Nano-base':>10} {'Nano-ft':>10} {'SV-ft':>10}")
print("-" * 70)

cers = [r['cer'] for r in all_results]
print(f"{'CER':<16} {cers[0]:>9.2%} {cers[1]:>9.2%} {cers[2]:>9.2%} {cers[3]:>9.2%}")

# 微调提升
sv_improve = (cers[0] - cers[3]) / cers[0] * 100 if cers[0] > 0 else 0
nano_improve = (cers[1] - cers[2]) / cers[1] * 100 if cers[1] > 0 else 0
print(f"{'微调提升':<16} {'':>10} {'':>10} {nano_improve:>+8.1f}% {sv_improve:>+8.1f}%")

exact_rates = [r['exact'] / max(r['total'], 1) for r in all_results]
print(f"{'精确匹配率':<16} {exact_rates[0]:>9.1%} {exact_rates[1]:>9.1%} {exact_rates[2]:>9.1%} {exact_rates[3]:>9.1%}")

speeds = [r['total'] / r['time'] for r in all_results]
print(f"{'速度(条/s)':<16} {speeds[0]:>9.1f} {speeds[1]:>9.1f} {speeds[2]:>9.1f} {speeds[3]:>9.1f}")

print(f"{'推理耗时':<16} {all_results[0]['time']:>8.1f}s {all_results[1]['time']:>8.1f}s {all_results[2]['time']:>8.1f}s {all_results[3]['time']:>8.1f}s")
print("=" * 70)

# 差异最大的样本（微调 vs 基线）
print(f"\n--- 微调效果最明显的样本 (CER 下降最多) ---")
sv_base_map = {r['id']: r for r in sv_base_res['results']}
sv_ft_map = {r['id']: r for r in sv_ft_res['results']}
nano_base_map = {r['id']: r for r in nano_base_res['results']}
nano_ft_map = {r['id']: r for r in nano_ft_res['results']}

common_ids = sorted(set(sv_base_map) & set(sv_ft_map) & set(nano_base_map) & set(nano_ft_map))

improvements = []
for idx in common_ids:
    sv_diff = sv_base_map[idx]['cer'] - sv_ft_map[idx]['cer']
    nano_diff = nano_base_map[idx]['cer'] - nano_ft_map[idx]['cer']
    improvements.append((idx, sv_diff, nano_diff))

improvements.sort(key=lambda x: max(x[1], x[2]), reverse=True)

print(f"{'ID':<5} {'期望':<15} {'SV基线':>7} {'SV微调':>7} {'Nano基线':>9} {'Nano微调':>9}")
print("-" * 65)
for idx, sv_d, nano_d in improvements[:10]:
    exp = sv_base_map[idx]['expected'][:12]
    print(f"{idx:<5} {exp:<15} {sv_base_map[idx]['cer']:>6.0%} {sv_ft_map[idx]['cer']:>6.0%} {nano_base_map[idx]['cer']:>8.0%} {nano_ft_map[idx]['cer']:>8.0%}")

## 5. 可视化

In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('ASR 四模型对比: 基线 vs 微调', fontsize=16, fontweight='bold')

names = ['SV-base', 'Nano-base', 'Nano-ft', 'SV-ft']
colors = ['#4ECDC4', '#45B7D1', '#FF6B6B', '#FF8E53']
cer_pcts = [r['cer'] * 100 for r in all_results]

# 1. CER 柱状图
bars = axes[0, 0].bar(names, cer_pcts, color=colors)
axes[0, 0].set_title('CER 对比')
axes[0, 0].set_ylabel('CER (%)')
for bar, cer in zip(bars, cer_pcts):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3, f'{cer:.1f}%', ha='center', va='bottom')

# 2. CER 分布直方图
for r, name, color in zip(all_results, names, colors):
    sample_cers = [s['cer'] * 100 for s in r['results']]
    axes[0, 1].hist(sample_cers, bins=20, alpha=0.5, label=name, color=color)
axes[0, 1].set_title('CER 分布')
axes[0, 1].set_xlabel('CER (%)')
axes[0, 1].set_ylabel('样本数')
axes[0, 1].legend()

# 3. SenseVoice: base vs ft 散点图
sv_base_cers = {s['id']: s['cer'] * 100 for s in sv_base_res['results']}
sv_ft_cers = {s['id']: s['cer'] * 100 for s in sv_ft_res['results']}
common_sv = sorted(set(sv_base_cers) & set(sv_ft_cers))
axes[1, 0].scatter([sv_base_cers[i] for i in common_sv], [sv_ft_cers[i] for i in common_sv], alpha=0.3, s=10, c='#4ECDC4')
max_cer = max(max(sv_base_cers.values()), max(sv_ft_cers.values()), 10)
axes[1, 0].plot([0, max_cer], [0, max_cer], 'r--', alpha=0.5, label='无变化线')
axes[1, 0].set_title('SenseVoice: 基线 vs 微调')
axes[1, 0].set_xlabel('基线 CER (%)')
axes[1, 0].set_ylabel('微调 CER (%)')
axes[1, 0].legend()

# 4. Nano: base vs ft 散点图
nano_base_cers = {s['id']: s['cer'] * 100 for s in nano_base_res['results']}
nano_ft_cers = {s['id']: s['cer'] * 100 for s in nano_ft_res['results']}
common_nano = sorted(set(nano_base_cers) & set(nano_ft_cers))
axes[1, 1].scatter([nano_base_cers[i] for i in common_nano], [nano_ft_cers[i] for i in common_nano], alpha=0.3, s=10, c='#45B7D1')
max_cer = max(max(nano_base_cers.values()), max(nano_ft_cers.values()), 10)
axes[1, 1].plot([0, max_cer], [0, max_cer], 'r--', alpha=0.5, label='无变化线')
axes[1, 1].set_title('Fun-ASR-Nano: 基线 vs 微调')
axes[1, 1].set_xlabel('基线 CER (%)')
axes[1, 1].set_ylabel('微调 CER (%)')
axes[1, 1].legend()

plt.tight_layout()
chart_path = "/mnt/output/model_comparison.png"
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图表已保存: {chart_path}")

In [ ]:
# 保存完整结果
compare_path = "/mnt/output/model_comparison.json"
with open(compare_path, 'w', encoding='utf-8') as f:
    json.dump({
        "models": {
            r['name']: {
                "cer": round(r['cer'], 4),
                "exact_match": r['exact'],
                "total": r['total'],
                "exact_rate": round(r['exact'] / max(r['total'], 1), 4),
                "time": round(r['time'], 1),
                "speed": round(r['total'] / r['time'], 1),
            } for r in all_results
        },
        "samples": [
            {"id": idx,
             "audio": valid_samples[idx]['source'],
             "expected": sv_base_map[idx]['expected'],
             "sv_base": sv_base_map[idx]['predicted'], "sv_base_cer": sv_base_map[idx]['cer'],
             "nano_base": nano_base_map[idx]['predicted'], "nano_base_cer": nano_base_map[idx]['cer'],
             "nano_ft": nano_ft_map[idx]['predicted'], "nano_ft_cer": nano_ft_map[idx]['cer'],
             "sv_ft": sv_ft_map[idx]['predicted'], "sv_ft_cer": sv_ft_map[idx]['cer']}
            for idx in common_ids
        ],
    }, f, ensure_ascii=False, indent=2)

print(f"对比结果已保存: {compare_path}")
print(f"图表已保存: {chart_path}")

## 6. 可视化网页报告

生成交互式 HTML 报告，支持：
- 音频播放 + 字符级 diff 高亮
- 按错误率筛选/排序
- 快速定位需要优化的录音样本

In [ ]:
from difflib import SequenceMatcher
import html as html_mod

# --- 字符级 diff ---
def char_diff(ref, hyp):
    sm = SequenceMatcher(None, ref, hyp)
    h_parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        t = html_mod.escape(hyp[j1:j2])
        if op == 'equal':
            h_parts.append(t)
        elif op == 'replace':
            h_parts.append(f'<span class="sub">{t}</span>')
        elif op == 'insert':
            h_parts.append(f'<span class="ins">{t}</span>')
    return ''.join(h_parts)

def cer_badge(c):
    if c == 0: return '<span class="b ok">✓</span>'
    if c < 0.15: return f'<span class="b warn">{c:.0%}</span>'
    if c < 0.4: return f'<span class="b bad">{c:.0%}</span>'
    return f'<span class="b vbad">{c:.0%}</span>'

# --- 生成数据 ---
REPORT_DIR = "/mnt/output"
rows_html = []
model_names = ['SV-base', 'Nano-base', 'Nano-ft', 'SV-ft']
model_maps = [sv_base_map, nano_base_map, nano_ft_map, sv_ft_map]

for idx in common_ids:
    audio_path = valid_samples[idx]['source']
    audio_rel = os.path.relpath(audio_path, REPORT_DIR)
    exp = sv_base_map[idx]['expected']

    cells = []
    for rmap in model_maps:
        pred = rmap[idx]['predicted']
        c = rmap[idx]['cer']
        diff = char_diff(exp, pred) if c > 0 else '<span class="ok-text">✓ 完全匹配</span>'
        cells.append(f'<td>{cer_badge(c)}<br><span class="t">{diff}</span></td>')

    worst = max(rmap[idx]['cer'] for rmap in model_maps)
    row_cls = 'error' if worst > 0 else 'match'
    audio_tag = f'<audio controls preload="none"><source src="{audio_rel}"></audio>'

    rows_html.append(
        f'<tr class="{row_cls}" data-c="{worst}">'
        f'<td class="id">{idx}</td>'
        f'<td>{audio_tag}</td>'
        f'<td class="ref">{html_mod.escape(exp)}</td>'
        + ''.join(cells) + '</tr>'
    )

# 摘要卡片
summary_cards = ""
for r in all_results:
    summary_cards += (
        f'<div class="card"><h3>{r["name"]}</h3>'
        f'<p class="big">{r["cer"]:.2%}</p>'
        f'<p>精确: {r["exact"]}/{r["total"]}</p></div>'
    )

html_report = f"""<!DOCTYPE html>
<html lang="zh-CN">
<head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>ASR 预测结果报告</title>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:-apple-system,"PingFang SC","Microsoft YaHei",sans-serif;background:#f0f2f5;color:#333;padding:20px}}
h1{{text-align:center;margin-bottom:20px;font-size:22px}}
.summary{{display:flex;gap:12px;justify-content:center;margin-bottom:20px;flex-wrap:wrap}}
.card{{background:#fff;padding:14px 22px;border-radius:8px;box-shadow:0 1px 3px rgba(0,0,0,.1);text-align:center;min-width:140px}}
.card h3{{font-size:12px;color:#888;margin-bottom:6px}}
.card .big{{font-size:22px;font-weight:bold;color:#1890ff}}
.card p{{font-size:12px;color:#666;margin-top:4px}}
.toolbar{{background:#fff;padding:12px 16px;border-radius:8px;margin-bottom:12px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;box-shadow:0 1px 3px rgba(0,0,0,.1)}}
.toolbar label{{font-size:13px;display:flex;align-items:center;gap:4px}}
.toolbar select,.toolbar input[type=range]{{cursor:pointer}}
.stats{{margin-left:auto;font-size:12px;color:#888}}
.wrap{{overflow-x:auto}}
table{{width:100%;border-collapse:collapse;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.1);font-size:13px}}
thead{{position:sticky;top:0;z-index:1}}
th{{background:#1a1a2e;color:#fff;padding:10px 8px;text-align:left;font-weight:500;white-space:nowrap}}
td{{padding:8px;border-bottom:1px solid #f0f0f0;vertical-align:top;max-width:280px;word-break:break-all}}
tr.error{{background:#fff5f5}}
tr.match{{background:#f6ffed}}
tr:hover{{filter:brightness(.97)}}
.id{{text-align:center;font-weight:bold;color:#666;min-width:40px}}
.ref{{font-weight:600;color:#333;min-width:80px}}
.t{{font-family:monospace;font-size:13px;line-height:1.8;display:inline-block}}
.b{{display:inline-block;padding:1px 6px;border-radius:3px;font-size:11px;font-weight:700;margin-bottom:2px}}
.b.ok{{background:#f6ffed;color:#52c41a;border:1px solid #b7eb8f}}
.b.warn{{background:#fffbe6;color:#faad14;border:1px solid #ffe58f}}
.b.bad{{background:#fff2f0;color:#ff4d4f;border:1px solid #ffccc7}}
.b.vbad{{background:#ff4d4f;color:#fff;border:1px solid #cf1322}}
.sub{{background:#ffe0b2;padding:0 1px;border-radius:2px;color:#e65100}}
.ins{{background:#c8e6c9;padding:0 1px;border-radius:2px;color:#1b5e20}}
.ok-text{{color:#52c41a;font-weight:600}}
audio{{height:30px;width:180px}}
</style>
</head>
<body>
<h1>ASR 预测结果对比报告</h1>
<div class="summary">{summary_cards}</div>
<div class="toolbar">
<label>筛选: <select id="filter" onchange="apply()">
<option value="all">全部</option>
<option value="error">仅错误</option>
<option value="match">仅匹配</option>
</select></label>
<label>最低 CER: <input id="thr" type="range" min="0" max="100" value="0" oninput="apply()"> <span id="tv">0%</span></label>
<label>排序: <select id="sort" onchange="apply()">
<option value="worst">最差优先</option>
<option value="id">按 ID</option>
</select></label>
<span class="stats" id="stats"></span>
</div>
<div class="wrap">
<table>
<thead><tr><th>ID</th><th>音频</th><th>期望</th><th>SV-base</th><th>Nano-base</th><th>Nano-ft</th><th>SV-ft</th></tr></thead>
<tbody id="tbody">{"".join(rows_html)}</tbody>
</table>
</div>
<script>
function apply(){{
  const f=document.getElementById('filter').value;
  const t=document.getElementById('thr').value/100;
  const s=document.getElementById('sort').value;
  document.getElementById('tv').textContent=Math.round(t*100)+'%';
  let rows=Array.from(document.querySelectorAll('#tbody tr'));
  let vis=0;
  rows.forEach(r=>{{
    const c=parseFloat(r.dataset.c);
    let show=true;
    if(f==='error'&&c===0)show=false;
    if(f==='match'&&c>0)show=false;
    if(c<t)show=false;
    r.style.display=show?'':'none';
    if(show)vis++;
  }});
  if(s==='worst')rows.sort((a,b)=>parseFloat(b.dataset.c)-parseFloat(a.dataset.c));
  else rows.sort((a,b)=>parseInt(a.querySelector('.id').textContent)-parseInt(b.querySelector('.id').textContent));
  const tb=document.getElementById('tbody');
  rows.forEach(r=>tb.appendChild(r));
  document.getElementById('stats').textContent='显示 '+vis+' / '+rows.length+' 条';
}}
apply();
</script>
</body>
</html>"""

report_path = f"{REPORT_DIR}/asr_report.html"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html_report)

print(f"网页报告已生成: {report_path}")
print(f"总样本数: {len(rows_html)}")
error_count = sum(1 for r in rows_html if 'class="error"' in r)
print(f"错误样本: {error_count} / {len(rows_html)}")
print()
print("查看方式:")
print(f"  cd /mnt && python -m http.server 8080")
print(f"  浏览器打开: http://<server-ip>:8080/output/asr_report.html")

## 7. 长音频测试

使用 VAD + SenseVoice-ft pipeline 测试 6 条长录音（10-30 分钟），评估真实连续语音表现。

In [ ]:
# --- 加载 VAD + SenseVoice-ft ---
try:
    import editdistance
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "editdistance"], check=True)
    import editdistance

PUNCT_PATTERN = re.compile(r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b]')

def normalize_text(text):
    return PUNCT_PATTERN.sub('', text)

def compute_cer(hyp, ref):
    hyp, ref = normalize_text(hyp), normalize_text(ref)
    if not ref: return 0.0
    return editdistance.eval(hyp, ref) / len(ref)

print("加载 VAD + SenseVoice-ft (长音频)...")
long_model = AutoModel(
    model="iic/SenseVoiceSmall", lora_only=True, disable_update=True,
    vad_model="iic/speech_fsmn_vad_zh-cn-16k-common-pytorch",
    vad_kwargs={"max_single_segment_time": 30000},
)
long_ckpt = torch.load(SV_CKPT, map_location="cpu")
long_model.model.load_state_dict(long_ckpt["state_dict"], strict=False)
print("模型加载成功!")

# --- 加载长音频数据 ---
LONG_TALKS_JSONL = "/mnt/data/hengdong_asr_trainset/manifests/extra_long_talks.jsonl"
long_samples = []
with open(LONG_TALKS_JSONL) as f:
    for line in f:
        item = json.loads(line)
        long_samples.append({
            'id': item['id'],
            'audio': item['audio_filepath'].replace(
                '/Users/fanhua/Desktop/hengdong_asr_trainset',
                '/mnt/data/hengdong_asr_trainset'
            ),
            'text': item['text'],
            'duration': item['duration'],
            'speaker': item.get('speaker_id', 'unknown'),
        })

print(f"长音频: {len(long_samples)} 条")
for s in long_samples:
    print(f"  {s['speaker']} | {s['duration']/60:.1f}min | {'✓' if os.path.exists(s['audio']) else '✗'}")

# --- 运行测试 ---
long_results = []
print(f"\n开始长音频测试...")
start_long = time.time()

for i, s in enumerate(long_samples):
    if not os.path.exists(s['audio']):
        print(f"  [{i+1}] 跳过: {s['audio']}")
        continue
    print(f"  [{i+1}/{len(long_samples)}] {s['speaker']} ({s['duration']/60:.1f}min)")
    result = long_model.generate(input=s['audio'], language="auto", use_itn=True)
    pred_text = "".join(r['text'] for r in result if 'text' in r)
    cer = compute_cer(pred_text, s['text'])
    long_results.append({
        'id': s['id'], 'speaker': s['speaker'], 'audio': s['audio'],
        'duration_min': s['duration'] / 60, 'cer': cer,
        'ref_len': len(normalize_text(s['text'])),
        'pred_len': len(normalize_text(pred_text)),
        'expected': s['text'], 'predicted': pred_text,
    })
    print(f"    CER: {cer:.1%} | 标注: {long_results[-1]['ref_len']}字 | 识别: {long_results[-1]['pred_len']}字")

long_elapsed = time.time() - start_long

# --- 汇总 ---
print(f"\n{'='*60}")
print(f"长音频测试汇总 (耗时 {long_elapsed:.1f}s)")
print(f"{'='*60}")
if long_results:
    avg_cer = sum(r['cer'] for r in long_results) / len(long_results)
    print(f"{'说话人':<20} {'时长':>6} {'CER':>8} {'标注字数':>8} {'识别字数':>8}")
    print("-" * 55)
    for r in long_results:
        print(f"{r['speaker']:<20} {r['duration_min']:>5.1f}m {r['cer']:>7.1%} {r['ref_len']:>8} {r['pred_len']:>8}")
    print("-" * 55)
    print(f"{'平均':<20} {'':>6} {avg_cer:>7.1%}")

# --- 保存 JSON ---
long_eval_path = f"{REPORT_DIR}/long_audio_eval.json"
with open(long_eval_path, 'w', encoding='utf-8') as f:
    json.dump({
        "model": "SenseVoice-ft (LoRA)", "total": len(long_results),
        "avg_cer": round(avg_cer, 4), "time": round(long_elapsed, 1),
        "results": long_results,
    }, f, ensure_ascii=False, indent=2)
print(f"结果已保存: {long_eval_path}")

# --- 生成长音频 HTML 报告 ---
long_rows = []
for r in long_results:
    audio_rel = os.path.relpath(r['audio'], REPORT_DIR)
    exp, pred, c = r['expected'], r['predicted'], r['cer']
    diff = char_diff(exp, pred) if c > 0 else '<span class="ok-text">✓ 完全匹配</span>'
    audio_tag = f'<audio controls preload="none"><source src="{audio_rel}"></audio>'
    long_rows.append(
        f'<tr class="{"error" if c > 0 else "match"}" data-c="{c}">'
        f'<td>{r["speaker"]}</td><td>{r["duration_min"]:.1f}min</td><td>{audio_tag}</td>'
        f'<td class="ref">{html_mod.escape(exp[:300])}</td>'
        f'<td>{cer_badge(c)}<br><span class="t">{diff[:500]}</span></td></tr>'
    )

long_html = f"""<!DOCTYPE html>
<html lang="zh-CN"><head><meta charset="UTF-8">
<title>长音频测试报告</title>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:-apple-system,"PingFang SC","Microsoft YaHei",sans-serif;background:#f0f2f5;color:#333;padding:20px}}
h1{{text-align:center;margin-bottom:20px;font-size:22px}}
.summary{{display:flex;gap:12px;justify-content:center;margin-bottom:20px;flex-wrap:wrap}}
.card{{background:#fff;padding:14px 22px;border-radius:8px;box-shadow:0 1px 3px rgba(0,0,0,.1);text-align:center;min-width:140px}}
.card h3{{font-size:12px;color:#888;margin-bottom:6px}}.card .big{{font-size:22px;font-weight:bold;color:#1890ff}}
.card p{{font-size:12px;color:#666;margin-top:4px}}
.wrap{{overflow-x:auto}}
table{{width:100%;border-collapse:collapse;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.1);font-size:13px}}
thead{{position:sticky;top:0;z-index:1}}th{{background:#1a1a2e;color:#fff;padding:10px 8px;text-align:left;font-weight:500;white-space:nowrap}}
td{{padding:8px;border-bottom:1px solid #f0f0f0;vertical-align:top;max-width:400px;word-break:break-all}}
tr.error{{background:#fff5f5}}tr.match{{background:#f6ffed}}tr:hover{{filter:brightness(.97)}}
.ref{{font-weight:600;color:#333}}.t{{font-family:monospace;font-size:13px;line-height:1.8;display:inline-block}}
.b{{display:inline-block;padding:1px 6px;border-radius:3px;font-size:11px;font-weight:700;margin-bottom:2px}}
.b.ok{{background:#f6ffed;color:#52c41a;border:1px solid #b7eb8f}}.b.warn{{background:#fffbe6;color:#faad14;border:1px solid #ffe58f}}
.b.bad{{background:#fff2f0;color:#ff4d4f;border:1px solid #ffccc7}}.b.vbad{{background:#ff4d4f;color:#fff;border:1px solid #cf1322}}
.sub{{background:#ffe0b2;padding:0 1px;border-radius:2px;color:#e65100}}.ins{{background:#c8e6c9;padding:0 1px;border-radius:2px;color:#1b5e20}}
.ok-text{{color:#52c41a;font-weight:600}}audio{{height:30px;width:180px}}
</style></head><body>
<h1>长音频测试报告 (SenseVoice-ft)</h1>
<div class="summary">
<div class="card"><h3>样本数</h3><p class="big">{len(long_results)}</p></div>
<div class="card"><h3>平均 CER</h3><p class="big">{avg_cer:.1%}</p></div>
<div class="card"><h3>总时长</h3><p class="big">{sum(r["duration_min"] for r in long_results):.1f}min</p></div>
</div>
<div class="wrap"><table>
<thead><tr><th>说话人</th><th>时长</th><th>音频</th><th>期望 (前300字)</th><th>预测</th></tr></thead>
<tbody>{"".join(long_rows)}</tbody></table></div></body></html>"""

long_report_path = f"{REPORT_DIR}/long_audio_report.html"
with open(long_report_path, 'w', encoding='utf-8') as f:
    f.write(long_html)
print(f"网页报告: {long_report_path}")

del long_model, long_ckpt
free_gpu()